# Proyecto Curso Intermedio de Analítica Avanzada

En este archivo se encuentran las pruebas realizadas para validar los modelos con sus parámetros optimizados. Este repositorio ha sido elaborado para poder analizar ciertas problemáticas de la NBA con modelos de machine learning y deep learning. Los autores de este proyecto son:

- Luis Alberto Arias Llaguno
- David Ceballos Mata
- Rubén Octavio Flores Ramos

En este archivo se verá problema por problema los resultados obtenidos para validar los modelos con sus parámetros optimizados. Asimismo, se verá la exploración de datos y la limpieza de datos de la api de la NBA para poder obtener los datos necesarios para los modelos.

# 0. Importación de librerías

In [ ]:
# Instalación de librerías

pip install nba_api

SyntaxError: invalid syntax (4211317163.py, line 4)

In [11]:
# Importación de librerías

from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams
from nba_api.stats.static import players
from nba_api.stats.endpoints import leaguedashplayerstats

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier

# 1. Exploración de datos

# 2. Problemas a Resolver

## 2.1 Predicción de rendimiento partido a partido usando secuencias de juegos (Luis)

### Problema: 
Predecir la línea de puntos, rebotes y asistencias de un jugador (en este caso LeBron James) en su próximo partido usando la secuencia de sus últimos N partidos, contexto del rival, viajes y carga de trabajo. Aquí la dependencia temporal es fuerte, y el comportamiento no es estacionario.

In [ ]:
nba_teams = teams.get_teams()
nba_players = players.get_active_players()

# Guardar en diccionarios id y nombre de equipos y jugadores
team_ids = {team['full_name']: team['id'] for team in nba_teams}
player_ids = {p['full_name']: p['id'] for p in nba_players}

# Imprimir id de equipos y jugadores
#print(team_ids)
#print(player_ids)

lebron_id = player_ids['LeBron James']
target = 25

gamefinder = leaguegamefinder.LeagueGameFinder(player_id_nullable=lebron_id)
games = gamefinder.get_data_frames()[0]
df_player = pd.DataFrame(games)

# Mostrar todas las columnas disponibles
#print(games.columns.tolist())

# Muestra columnas útiles
#print(games[['GAME_DATE', 'MATCHUP', 'WL', 'PTS', 'REB', 'AST', 'TOV', 'FG_PCT', 'FG3_PCT']].head())
df_player = df_player[['TEAM_ABBREVIATION', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']]

# Eliminar filas con ANY valor nulo en df_player, que en realidad son pocas 
df_player = df_player.dropna().reset_index(drop=True)

df_player.head()

df_player.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1875 entries, 0 to 1874
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   TEAM_ABBREVIATION  1875 non-null   object 
 1   GAME_DATE          1875 non-null   object 
 2   MATCHUP            1875 non-null   object 
 3   WL                 1875 non-null   object 
 4   MIN                1875 non-null   int64  
 5   PTS                1875 non-null   int64  
 6   FG_PCT             1875 non-null   float64
 7   FG3_PCT            1875 non-null   float64
 8   FT_PCT             1875 non-null   float64
 9   OREB               1875 non-null   int64  
 10  DREB               1875 non-null   int64  
 11  REB                1875 non-null   int64  
 12  AST                1875 non-null   int64  
 13  STL                1875 non-null   int64  
 14  BLK                1875 non-null   int64  
 15  TOV                1875 non-null   int64  
 16  PF                 1875 

In [16]:
from nba_api.stats.endpoints import PlayerGameLog

lebron_id = player_ids['LeBron James']
target = 25

gamefinder = leaguegamefinder.LeagueGameFinder(player_id_nullable=lebron_id)
games = gamefinder.get_data_frames()[0]
df_player = pd.DataFrame(games)

df_player.columns = df_player.columns.str.lower().str.replace(' ', '_')
df_player['game_date'] = pd.to_datetime(df_player['game_date'])
df_player = df_player.sort_values(['lebron_id', 'game_date'])
df_player = df_player.drop_duplicates(subset=['lebron_id', 'game_id'])
df_player['days_rest'] = df_player.groupby('lebron_id')['game_date'].diff().dt.days
df_player['days_rest'] = df_player['days_rest'].fillna(df_player['days_rest'].median())
df_player['is_home'] = df_player['matchup'].str.contains('vs').astype(int)
df_player['opponent'] = df_player['matchup'].str.split(' ').str[-1]
stats = ['pts', 'reb', 'ast', 'min']
for col in stats:
    df_player[f'{col}_mean5'] = df_player.groupby('lebron_id')[col].rolling(5).mean().reset_index(0, drop=True)
    df_player[f'{col}_std5'] = df_player.groupby('lebron_id')[col].rolling(5).std().reset_index(0, drop=True)
df = df.fillna(df.median(numeric_only=True))
def build_sequences(df, seq_len=10):
    X, y = [], []
    for _, player_df in df.groupby('lebron_id'):
        values = player_df[feature_cols].values
        targets = player_df[target_col].values
        
        for i in range(len(values) - seq_len):
            X.append(values[i:i+seq_len])
            y.append(targets[i+seq_len])
    
    return np.array(X), np.array(y)




KeyError: 'lebron_id'